# Analysis of the data set 

In [ ]:
import os

os.environ["JAVA_HOME"] = r"C:\Users\axel.potier\AppData\Local\anaconda3\envs\env_axel\Library"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Analyse")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

In [ ]:
csv_path = "../data/postings.csv"

jobs = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)   # certaines lignes contiennent des retours à la ligne dans "description"
    .option("quote", '"')
    .option("escape", '"')
    .csv(csv_path)
)

jobs.show(5, truncate=False)
jobs.printSchema()

In [ ]:
jobs.createOrReplaceTempView("postings")

result = spark.sql("""
    SELECT company_name, AVG(normalized_salary) AS avg_salary, COUNT(*) AS nb_offres
    FROM postings
    WHERE normalized_salary IS NOT NULL
      AND company_name IS NOT NULL
    GROUP BY company_name
    HAVING COUNT(*) >= 3
    ORDER BY avg_salary DESC
    LIMIT 10
""")
result.show()